# Face Recognition Server
**Chạy trên Google Colab — phục vụ app điểm danh**

## Hướng dẫn sử dụng

| Cell | Nội dung | Chạy lại khi nào |
|------|----------|------------------|
| 1 | Mount Google Drive | Mỗi session |
| 2 | Cài dependencies + tải model | Lần đầu / sau reset runtime |
| 3 | Ghi code server lên Colab | Khi thay đổi code |
| 4 | Khởi động server + ngrok | Mỗi session |
| 5 | Kiểm tra nhanh | Tùy chọn |

**Khi restart session:** chỉ cần chạy Cell 1 → Cell 4. Dữ liệu trên Drive không mất.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ nếu chưa có
import os
os.makedirs('/content/drive/MyDrive/attendance', exist_ok=True)
print('Drive đã mount. Thư mục attendance sẵn sàng.')

## Cell 2 — Cài dependencies & tải InsightFace model
**Chỉ cần chạy 1 lần per session (hoặc sau khi reset runtime)**

In [ ]:
# Cài tất cả dependencies
!pip install -q \
    fastapi==0.110.2 \
    "uvicorn[standard]==0.29.0" \
    python-multipart==0.0.9 \
    pyngrok==7.1.6 \
    insightface==0.7.3 \
    onnxruntime==1.17.3 \
    opencv-python-headless==4.9.0.80 \
    numpy==1.26.4 \
    "pydantic==2.7.0"

print('\n--- Kiểm tra InsightFace model ---')
# Tải model buffalo_l (~300 MB, chỉ tải 1 lần, cache lại sau đó)
import insightface
from insightface.app import FaceAnalysis
_check = FaceAnalysis(name='buffalo_l')
_check.prepare(ctx_id=0, det_size=(640, 640))
del _check
print('Model buffalo_l đã sẵn sàng (cache tại ~/.insightface/).')

## Cell 3 — Ghi code server
**Chạy lại cell này khi bạn thay đổi code**

In [ ]:
import os
os.makedirs('/content/server', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

In [ ]:
%%writefile /content/server/config.py
# ============================================================
# CẤU HÌNH — chỉnh sửa tại đây trước khi chạy server
# ============================================================

# Đường dẫn file database trên Google Drive
DRIVE_DB_PATH = "/content/drive/MyDrive/attendance/face_db.json"

# Fallback khi Drive chưa mount
LOCAL_DB_PATH = "/content/data/face_db.json"

# Threshold cosine distance (0.4 = trong nhà, 0.45-0.5 = outdoor)
DEFAULT_THRESHOLD = 0.4

# Kích thước ảnh input cho detection
DET_SIZE = (640, 640)

# Model InsightFace
INSIGHTFACE_MODEL = "buffalo_l"

# ⚠️ QUAN TRỌNG: Đổi API key này trước khi deploy!
# Chia sẻ key này cho developer app điểm danh.
API_KEY = "attendance-app-secret-key-2026"

# Số ảnh tối đa khi đăng ký
MAX_REGISTER_IMAGES = 20

In [ ]:
%%writefile /content/server/database.py
"""
Quản lý dữ liệu khuôn mặt — đọc/ghi face_db.json trên Google Drive.
"""

import json
import logging
import os
from typing import Optional

import numpy as np

from server.config import DRIVE_DB_PATH, LOCAL_DB_PATH

logger = logging.getLogger(__name__)


def _get_db_path() -> str:
    drive_dir = os.path.dirname(DRIVE_DB_PATH)
    if os.path.isdir(drive_dir):
        return DRIVE_DB_PATH
    logger.warning("Drive chưa mount, dùng local: %s", LOCAL_DB_PATH)
    return LOCAL_DB_PATH


def load_db() -> dict:
    path = _get_db_path()
    if not os.path.exists(path):
        logger.info("DB chưa tồn tại tại %s, khởi tạo rỗng.", path)
        return {}
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    db = {}
    for pid, rec in raw.items():
        db[pid] = {
            "name": rec["name"],
            "embedding": np.array(rec["embedding"], dtype=np.float32),
            "registered_at": rec["registered_at"],
        }
    logger.info("Loaded %d người từ %s", len(db), path)
    return db


def save_db(db: dict) -> None:
    path = _get_db_path()
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    serializable = {}
    for pid, rec in db.items():
        emb = rec["embedding"]
        serializable[pid] = {
            "name": rec["name"],
            "embedding": emb.tolist() if isinstance(emb, np.ndarray) else emb,
            "registered_at": rec["registered_at"],
        }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)
    logger.info("Saved %d người vào %s", len(db), path)


def upsert_person(db, person_id, name, embedding, registered_at):
    db[person_id] = {"name": name, "embedding": embedding, "registered_at": registered_at}
    save_db(db)


def delete_person(db: dict, person_id: str) -> bool:
    if person_id not in db:
        return False
    del db[person_id]
    save_db(db)
    return True


def get_all_persons(db: dict) -> list:
    return [
        {"person_id": pid, "name": rec["name"], "registered_at": rec["registered_at"]}
        for pid, rec in db.items()
    ]


def build_embeddings_matrix(db: dict) -> tuple:
    if not db:
        return None, []
    ids = list(db.keys())
    matrix = np.stack([db[pid]["embedding"] for pid in ids]).astype(np.float32)
    return matrix, ids

In [ ]:
%%writefile /content/server/face_service.py
"""
InsightFace wrapper — phát hiện, căn chỉnh, nhúng và so sánh khuôn mặt.
"""

import base64
import logging
from typing import Optional

import cv2
import numpy as np
from insightface.app import FaceAnalysis

from server.config import DET_SIZE, INSIGHTFACE_MODEL

logger = logging.getLogger(__name__)


class FaceService:
    def __init__(self) -> None:
        logger.info("Đang tải InsightFace model '%s'...", INSIGHTFACE_MODEL)
        self.analyzer = FaceAnalysis(name=INSIGHTFACE_MODEL)
        self.analyzer.prepare(ctx_id=0, det_size=DET_SIZE)
        logger.info("InsightFace model đã sẵn sàng.")

    def decode_image(self, b64_string: str) -> Optional[np.ndarray]:
        try:
            if "," in b64_string:
                b64_string = b64_string.split(",", 1)[1]
            img_bytes = base64.b64decode(b64_string)
            img_array = np.frombuffer(img_bytes, dtype=np.uint8)
            return cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        except Exception as exc:
            logger.warning("Không decode được ảnh: %s", exc)
            return None

    def get_embedding(self, image: np.ndarray) -> Optional[np.ndarray]:
        faces = self.analyzer.get(image)
        if not faces:
            return None
        # Chọn khuôn mặt lớn nhất
        largest = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
        return largest.embedding.astype(np.float32)

    def average_embeddings(self, embeddings: list) -> np.ndarray:
        stacked = np.stack(embeddings, axis=0)
        mean_emb = stacked.mean(axis=0)
        norm = np.linalg.norm(mean_emb)
        if norm > 0:
            mean_emb = mean_emb / norm
        return mean_emb.astype(np.float32)

    @staticmethod
    def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
        return float(1.0 - np.dot(a, b))

    def find_best_match(self, query_embedding, embeddings_matrix, person_ids, db, threshold):
        if embeddings_matrix is None or not person_ids:
            return None, None, 1.0
        distances = 1.0 - embeddings_matrix @ query_embedding
        min_idx = int(np.argmin(distances))
        min_distance = float(distances[min_idx])
        if min_distance <= threshold:
            pid = person_ids[min_idx]
            return pid, db[pid]["name"], min_distance
        return None, None, min_distance

In [ ]:
%%writefile /content/server/models.py
"""
Pydantic v2 schemas cho request/response.
"""

from typing import Literal, Optional
from pydantic import BaseModel, Field, model_validator
from server.config import DEFAULT_THRESHOLD, MAX_REGISTER_IMAGES


class RegisterRequest(BaseModel):
    person_id: str = Field(..., min_length=1, max_length=50)
    name: str = Field(..., min_length=1, max_length=100)
    images: list[str] = Field(..., min_length=1, max_length=MAX_REGISTER_IMAGES)

    @model_validator(mode="after")
    def check_images_not_empty(self):
        for i, img in enumerate(self.images):
            if not img.strip():
                raise ValueError(f"images[{i}] không được rỗng")
        return self


class RecognizeRequest(BaseModel):
    image: str
    threshold: float = Field(DEFAULT_THRESHOLD, ge=0.1, le=1.0)


class RegisterResponse(BaseModel):
    status: Literal["ok", "error"]
    person_id: str
    name: str
    embedding: list[float]
    faces_detected: int
    faces_failed: int


class RecognizeResponse(BaseModel):
    status: Literal["recognized", "unknown", "no_face"]
    person_id: Optional[str] = None
    name: Optional[str] = None
    confidence: Optional[float] = None
    distance: Optional[float] = None


class PersonInfo(BaseModel):
    person_id: str
    name: str
    registered_at: str


class PersonsListResponse(BaseModel):
    persons: list[PersonInfo]


class DeleteResponse(BaseModel):
    status: Literal["ok", "not_found"]
    deleted: Optional[str] = None


class HealthResponse(BaseModel):
    status: Literal["ok"]
    model_loaded: bool
    registered_persons: int


class ErrorResponse(BaseModel):
    status: Literal["error"]
    detail: str

In [ ]:
%%writefile /content/server/main.py
"""
FastAPI application — điểm vào chính.
"""

import logging
from contextlib import asynccontextmanager
from datetime import datetime, timezone

from fastapi import FastAPI, HTTPException, Request, Security
from fastapi.security.api_key import APIKeyHeader

from server.config import API_KEY
from server.database import (
    build_embeddings_matrix, delete_person, get_all_persons,
    load_db, upsert_person,
)
from server.face_service import FaceService
from server.models import (
    DeleteResponse, HealthResponse, PersonsListResponse,
    RecognizeRequest, RecognizeResponse,
    RegisterRequest, RegisterResponse, ErrorResponse,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger(__name__)

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def verify_api_key(api_key: str = Security(api_key_header)) -> str:
    if not api_key or api_key != API_KEY:
        raise HTTPException(status_code=401, detail="API key không hợp lệ hoặc thiếu header X-API-Key")
    return api_key


@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("=== Server đang khởi động ===")
    app.state.face_service = FaceService()
    app.state.db = load_db()
    matrix, ids = build_embeddings_matrix(app.state.db)
    app.state.embeddings_matrix = matrix
    app.state.person_ids = ids
    logger.info("=== Server sẵn sàng. Đã load %d người. ===", len(app.state.db))
    yield
    logger.info("=== Server đang tắt ===")


app = FastAPI(
    title="Face Recognition Server",
    description="Server nhận diện khuôn mặt (InsightFace/ArcFace) chạy trên Google Colab.",
    version="2.0.0",
    lifespan=lifespan,
)


def _rebuild_matrix(request: Request) -> None:
    matrix, ids = build_embeddings_matrix(request.app.state.db)
    request.app.state.embeddings_matrix = matrix
    request.app.state.person_ids = ids


@app.get("/health", response_model=HealthResponse, tags=["Utility"])
async def health(request: Request):
    """Kiểm tra trạng thái server — không cần API key."""
    return HealthResponse(
        status="ok",
        model_loaded=hasattr(request.app.state, "face_service"),
        registered_persons=len(request.app.state.db),
    )


@app.post("/register", response_model=RegisterResponse,
          responses={422: {"model": ErrorResponse}}, tags=["Face"])
async def register(body: RegisterRequest, request: Request,
                   _key: str = Security(verify_api_key)):
    """
    Đăng ký khuôn mặt mới (hoặc cập nhật nếu person_id đã tồn tại).
    Gửi 3-10 ảnh để đạt độ chính xác tốt nhất.
    """
    face_service: FaceService = request.app.state.face_service
    embeddings, failed = [], 0

    for b64_img in body.images:
        img = face_service.decode_image(b64_img)
        if img is None:
            failed += 1
            continue
        emb = face_service.get_embedding(img)
        if emb is None:
            failed += 1
        else:
            embeddings.append(emb)

    if not embeddings:
        raise HTTPException(
            status_code=422,
            detail="Không detect được khuôn mặt trong bất kỳ ảnh nào. Hãy chụp lại với ánh sáng đủ.",
        )

    representative_emb = face_service.average_embeddings(embeddings)
    registered_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
    upsert_person(request.app.state.db, body.person_id, body.name, representative_emb, registered_at)
    _rebuild_matrix(request)

    logger.info("Đã đăng ký %s (%s): %d/%d ảnh thành công",
                body.person_id, body.name, len(embeddings), len(body.images))

    return RegisterResponse(
        status="ok",
        person_id=body.person_id,
        name=body.name,
        embedding=representative_emb.tolist(),
        faces_detected=len(embeddings),
        faces_failed=failed,
    )


@app.post("/recognize", response_model=RecognizeResponse, tags=["Face"])
async def recognize(body: RecognizeRequest, request: Request,
                    _key: str = Security(verify_api_key)):
    """
    Nhận diện khuôn mặt từ 1 ảnh.
    status: recognized | unknown | no_face
    """
    face_service: FaceService = request.app.state.face_service
    img = face_service.decode_image(body.image)
    if img is None:
        raise HTTPException(status_code=422, detail="Không decode được ảnh.")

    emb = face_service.get_embedding(img)
    if emb is None:
        return RecognizeResponse(status="no_face")

    pid, name, distance = face_service.find_best_match(
        emb, request.app.state.embeddings_matrix,
        request.app.state.person_ids, request.app.state.db, body.threshold,
    )

    if pid is not None:
        return RecognizeResponse(
            status="recognized", person_id=pid, name=name,
            confidence=round(1.0 - distance, 4), distance=round(distance, 4),
        )
    return RecognizeResponse(status="unknown", distance=round(distance, 4))


@app.get("/persons", response_model=PersonsListResponse, tags=["Persons"])
async def list_persons(request: Request, _key: str = Security(verify_api_key)):
    """Lấy danh sách tất cả người đã đăng ký."""
    return PersonsListResponse(persons=get_all_persons(request.app.state.db))


@app.delete("/persons/{person_id}", response_model=DeleteResponse, tags=["Persons"])
async def remove_person(person_id: str, request: Request,
                        _key: str = Security(verify_api_key)):
    """Xóa 1 người khỏi database."""
    deleted = delete_person(request.app.state.db, person_id)
    if not deleted:
        return DeleteResponse(status="not_found")
    _rebuild_matrix(request)
    logger.info("Đã xóa person_id: %s", person_id)
    return DeleteResponse(status="ok", deleted=person_id)

In [ ]:
# Tạo __init__.py để Python nhận server/ là package
open('/content/server/__init__.py', 'w').close()
print('Tất cả file server đã được ghi xong.')

## Cell 4 — Khởi động Server + ngrok
**Chạy lại mỗi session. Sau khi chạy, copy URL ở output vào app điểm danh.**

In [ ]:
import subprocess
import sys
import threading
import time

from pyngrok import ngrok

# ─────────────────────────────────────────────────────────
# ⚙️  CẤU HÌNH: Paste ngrok auth token của bạn vào đây
# Lấy token miễn phí tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "paste-your-ngrok-token-here"
# ─────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Tắt ngrok tunnel cũ nếu đang chạy
ngrok.kill()

# Khởi động uvicorn trong background thread
def run_server():
    subprocess.run(
        [sys.executable, "-m", "uvicorn", "server.main:app",
         "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
        cwd="/content"
    )

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Đợi server khởi động
time.sleep(5)

# Mở tunnel ngrok
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url

print()
print('=' * 60)
print(f'  PUBLIC URL : {public_url}')
print(f'  Swagger UI : {public_url}/docs')
print(f'  Health     : {public_url}/health')
print('=' * 60)
print()
print('⚠️  Copy URL trên vào phần cấu hình của app điểm danh.')
print('    URL thay đổi mỗi lần khởi động lại — nhớ cập nhật!')

## Cell 5 — Kiểm tra nhanh (tùy chọn)
Chạy sau Cell 4 để xác nhận server hoạt động đúng.

In [ ]:
import requests
import base64
import json

# Thay bằng URL từ Cell 4
BASE_URL = public_url  # hoặc paste trực tiếp: "https://xxxx.ngrok-free.app"
API_KEY = "attendance-app-secret-key-2026"  # phải khớp với config.py
HEADERS = {"X-API-Key": API_KEY}

# 1. Health check
r = requests.get(f"{BASE_URL}/health")
print("Health check:", json.dumps(r.json(), ensure_ascii=False, indent=2))

# 2. Danh sách người đã đăng ký
r = requests.get(f"{BASE_URL}/persons", headers=HEADERS)
persons = r.json()
print(f"\nĐã đăng ký: {len(persons['persons'])} người")
for p in persons['persons']:
    print(f"  - {p['person_id']}: {p['name']} ({p['registered_at']})")

# 3. Test đăng ký với ảnh webcam (bỏ comment nếu muốn test)
# from IPython.display import display
# from google.colab.patches import cv2_imshow
#
# Đọc 1 ảnh từ file và convert sang base64 để test:
# with open("/path/to/test.jpg", "rb") as f:
#     b64_img = base64.b64encode(f.read()).decode()
#
# payload = {
#     "person_id": "TEST001",
#     "name": "Nguyen Test",
#     "images": [b64_img]
# }
# r = requests.post(f"{BASE_URL}/register", json=payload, headers=HEADERS)
# print("Register:", r.json())
#
# payload_rec = {"image": b64_img}
# r = requests.post(f"{BASE_URL}/recognize", json=payload_rec, headers=HEADERS)
# print("Recognize:", r.json())